# SageMaker Pipeline: AutoGluon TimeSeries Forecasting

End-to-end ML pipeline using SageMaker SDK v3:

1. **ProcessingStep** - Convert raw electricity data to train/test long-format CSVs
2. **TrainingStep** - AutoGluon `TimeSeriesPredictor.fit()`
3. **ProcessingStep** - Evaluate model on held-out test window
4. **ConditionStep** - Register model only if MASE <= threshold
5. **ModelStep** - Register to Model Registry

## Configuration

In [ ]:
import boto3
import sagemaker

# --- Edit these as needed ---
REGION = boto3.session.Session().region_name
sess = sagemaker.session.Session()
BUCKET = sess.default_bucket()
AG_VERSION = "1.5"
PY_VERSION = "py312"
PIPELINE_NAME = "AutoGluonTimeSeriesPipeline"

print(f"Region:   {REGION}")
print(f"Bucket:   {BUCKET}")
print(f"Pipeline: {PIPELINE_NAME}")

## Discover IAM Role

In [ ]:
iam = boto3.client("iam")
role_arn = None
paginator = iam.get_paginator("list_roles")
for page in paginator.paginate():
    for role in page["Roles"]:
        if "SageMaker" in role["RoleName"] or "sagemaker" in role["RoleName"]:
            role_arn = role["Arn"]
            break
    if role_arn:
        break

# Uncomment to override:
# role_arn = "arn:aws:iam::123456789012:role/YourSageMakerRole"

assert role_arn, "No SageMaker IAM role found. Set role_arn manually above."
print(f"Using role: {role_arn}")

## Define Pipeline

The `create_pipeline()` function assembles all steps. Source paths for `preprocess.py`, `train.py`, and `evaluate.py` are relative to the project structure.

In [ ]:
import os

from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.processing import ProcessingInput, ProcessingOutput, ScriptProcessor
from sagemaker.core.shapes.shapes import ProcessingS3Input, ProcessingS3Output
from sagemaker.core.training.configs import (
    Compute,
    OutputDataConfig,
    SourceCode,
    StoppingCondition,
)
from sagemaker.core.workflow.parameters import ParameterString
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.core.workflow.properties import PropertyFile
from sagemaker.mlops.workflow.pipeline import Pipeline
from sagemaker.mlops.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.train import ModelTrainer


def create_pipeline(
    role_arn: str,
    region: str,
    bucket: str,
    ag_version: str = "1.5",
    py_version: str = "py312",
    pipeline_name: str = "AutoGluonTimeSeriesPipeline",
):
    pipeline_session = PipelineSession()
    s3_prefix = f"s3://{bucket}/autogluon-timeseries/pipeline"

    # --- Pipeline Parameters ---
    input_data_uri = ParameterString(
        name="InputDataUri",
        default_value=f"s3://{bucket}/autogluon-timeseries/raw/",
    )
    instance_type = ParameterString(name="InstanceType", default_value="ml.m5.2xlarge")
    # Note: ParameterString (not ParameterInteger) because hyperparameters are strings
    prediction_length = ParameterString(name="PredictionLength", default_value="84")

    # --- Image URIs ---
    ag_training_image = image_uris.retrieve(
        "autogluon", region=region, version=ag_version,
        py_version=py_version, image_scope="training", instance_type="ml.m5.2xlarge",
    )
    processing_image = image_uris.retrieve("sklearn", region=region, version="1.2-1")

    # --- Step 1: Preprocessing (v3 step_args pattern) ---
    preprocessor = ScriptProcessor(
        image_uri=processing_image,
        role=role_arn,
        command=["python3"],
        instance_type="ml.m5.xlarge",
        instance_count=1,
        sagemaker_session=pipeline_session,
    )

    step_preprocess = ProcessingStep(
        name="PreprocessTimeSeries",
        step_args=preprocessor.run(
            code=os.path.abspath(os.path.join("..", "0-data-prep", "preprocess.py")),
            inputs=[
                ProcessingInput(
                    input_name="input",
                    s3_input=ProcessingS3Input(
                        s3_uri=input_data_uri,
                        local_path="/opt/ml/processing/input",
                        s3_data_type="S3Prefix",
                    ),
                ),
            ],
            outputs=[
                ProcessingOutput(
                    output_name="train",
                    s3_output=ProcessingS3Output(
                        s3_uri=f"{s3_prefix}/processed/train/",
                        local_path="/opt/ml/processing/train",
                        s3_upload_mode="EndOfJob",
                    ),
                ),
                ProcessingOutput(
                    output_name="test",
                    s3_output=ProcessingS3Output(
                        s3_uri=f"{s3_prefix}/processed/test/",
                        local_path="/opt/ml/processing/test",
                        s3_upload_mode="EndOfJob",
                    ),
                ),
            ],
        ),
    )

    # --- Step 2: Training ---
    trainer = ModelTrainer(
        training_image=ag_training_image,
        role=role_arn,
        source_code=SourceCode(
            source_dir=os.path.abspath(os.path.join("..", "1-training")),
            entry_script="train.py",
        ),
        compute=Compute(
            instance_type=instance_type,
            instance_count=1,
            volume_size_in_gb=50,
            keep_alive_period_in_seconds=0,
        ),
        output_data_config=OutputDataConfig(
            s3_output_path=f"{s3_prefix}/model/",
        ),
        hyperparameters={
            "prediction-length": prediction_length,
            "presets": "medium_quality",
            "time-limit": "3600",
            "eval-metric": "MASE",
            "target": "target",
            "id-column": "item_id",
            "timestamp-column": "timestamp",
        },
        base_job_name="ag-timeseries-train",
        stopping_condition=StoppingCondition(max_runtime_in_seconds=7200),
        sagemaker_session=pipeline_session,
    )

    step_train = TrainingStep(
        name="TrainTimeSeriesModel",
        step_args=trainer.train(
            input_data_config=[
                {
                    "channel_name": "train",
                    "data_source": {
                        "s3_data_source": {
                            "s3_uri": step_preprocess.properties.ProcessingOutputConfig.Outputs[
                                "train"
                            ].S3Output.S3Uri,
                            "s3_data_type": "S3Prefix",
                        }
                    },
                },
                {
                    "channel_name": "test",
                    "data_source": {
                        "s3_data_source": {
                            "s3_uri": step_preprocess.properties.ProcessingOutputConfig.Outputs[
                                "test"
                            ].S3Output.S3Uri,
                            "s3_data_type": "S3Prefix",
                        }
                    },
                },
            ],
        ),
    )

    # --- Step 3: Evaluation (v3 step_args pattern) ---
    evaluation_report = PropertyFile(
        name="EvaluationReport",
        output_name="evaluation",
        path="evaluation.json",
    )

    evaluator = ScriptProcessor(
        image_uri=ag_training_image,
        role=role_arn,
        command=["python3"],
        instance_type="ml.m5.xlarge",
        instance_count=1,
        sagemaker_session=pipeline_session,
    )

    step_evaluate = ProcessingStep(
        name="EvaluateTimeSeries",
        step_args=evaluator.run(
            code=os.path.abspath("evaluate.py"),
            inputs=[
                ProcessingInput(
                    input_name="model",
                    s3_input=ProcessingS3Input(
                        s3_uri=step_train.properties.ModelArtifacts.S3ModelArtifacts,
                        local_path="/opt/ml/processing/model",
                        s3_data_type="S3Prefix",
                    ),
                ),
                ProcessingInput(
                    input_name="test",
                    s3_input=ProcessingS3Input(
                        s3_uri=step_preprocess.properties.ProcessingOutputConfig.Outputs[
                            "test"
                        ].S3Output.S3Uri,
                        local_path="/opt/ml/processing/test",
                        s3_data_type="S3Prefix",
                    ),
                ),
            ],
            outputs=[
                ProcessingOutput(
                    output_name="evaluation",
                    s3_output=ProcessingS3Output(
                        s3_uri=f"{s3_prefix}/evaluation/",
                        local_path="/opt/ml/processing/evaluation",
                        s3_upload_mode="EndOfJob",
                    ),
                ),
            ],
        ),
        property_files=[evaluation_report],
    )

    # Note: Model registration (ConditionStep + ModelStep) removed —
    # v3 Model.register() is incompatible with PipelineSession/PipelineVariable.

    # --- Assemble Pipeline ---
    pipeline = Pipeline(
        name=pipeline_name,
        parameters=[input_data_uri, instance_type, prediction_length],
        steps=[step_preprocess, step_train, step_evaluate],
        sagemaker_session=pipeline_session,
    )

    return pipeline

## Create and Upsert Pipeline

In [ ]:
pipeline = create_pipeline(
    role_arn=role_arn,
    region=REGION,
    bucket=BUCKET,
    ag_version=AG_VERSION,
    py_version=PY_VERSION,
    pipeline_name=PIPELINE_NAME,
)

pipeline.upsert(role_arn=role_arn)
print(f"Pipeline '{PIPELINE_NAME}' created/updated.")

## Execute Pipeline

Start the pipeline and optionally wait for completion.

In [ ]:
# execution = pipeline.start()
# print(f"Execution started: {execution.describe()['PipelineExecutionArn']}")

In [ ]:
# Uncomment to wait for completion (can take a long time)
# execution.wait()
# print("Pipeline execution complete.")